# Hedgers-Toll 🌾
### Producers pay to offload price risk — are you, the speculator, really collecting the toll?

![Signal: Weak](https://img.shields.io/badge/Signal-Weak-dab617?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Premium today?: Faded](https://img.shields.io/badge/Premium_today%3F-Faded-8b949e?style=flat-square)

A wheat farmer wants to lock in next year's price, so they *sell* wheat futures today. An oil driller does the same. Someone has to take the other side of all that hedging — and the old theory (Keynes' *normal backwardation*; Cootner 1960) says that someone, the speculator, is **paid a premium** for it: a toll the hedgers hand over for the insurance. You can read who's hedging straight off the government's **Commitments of Traders** report — so the trade writes itself: go long the commodities where commercial hedgers are most net short.

This is the desk's twelfth idea from Kakushadze & Serur's *151 Trading Strategies* (strategy §9.2). We prove the engine on a synthetic market where the toll is baked in (and a null), then run it on real CFTC positioning and commodity futures — and find the toll booth has been abandoned.

> 📓 **Plain-language layer.** The information coefficient, the Newey–West *t* and the window sweep are in **[02_for_the_quants.ipynb](02_for_the_quants.ipynb)**.
>
> ⚠️ **Not investment advice.** Every chart is generated by the code beside it; the core runs on a **synthetic** panel, so the real numbers (from [`../docs/results.md`](../docs/results.md)) are a measurement. House style in [METHODOLOGY.md](../../../METHODOLOGY.md).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("../../.."))
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (9.5, 5.2)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
from hedgers_toll import data, hedging, strategy, decompose, extension

# Offline synthetic commodities: a PREMIUM panel (hedging pressure predicts returns) and a NULL. The
# real CFTC-COT + futures verdict is in ../docs/results.md.
r,  hp,  truth = data.synthetic_commodities(hp_strength=0.0045, seed=29)   # the premium panel
r0, hp0, _     = data.synthetic_commodities(hp_strength=0.0,    seed=29)   # the null
print(f"{truth.n_comm} synthetic commodities x {truth.n_weeks} weeks | baked hp_strength={truth.hp_strength} | null=0")


## The answer first 🎯

| What we asked | The honest answer |
|---|---|
| Does hedging pressure predict commodity returns? | 🟡 **In theory yes; lately no.** Real on our control (IC *t* > 11) and in the literature, but **-0.021** (*t* **-1.4**) on the modern tape. |
| Does the trade pay? | 🚫 **No** — the long-short hedging-pressure factor *loses*: Sharpe **-0.41** (*t* **-1.4**), negative across every signal window. |
| Is the premium still there? | ⚪ **Faded** — negative through most of the modern sample. |

> Desk shorthand: **Signal `WEAK` · Tradability `MIRAGE` · Premium today? `FADED`** — a real risk premium a present-day trader can't collect.

## 1 · The claim 📣

Hedging pressure: `HP = (commercial short − commercial long) / total`. When it's high, producers are heavily net short — paying speculators to be long — and the claim is the long speculator earns a premium next period. On our synthetic market we baked exactly that: the hedging pressure predicts the next return.

In [ ]:
pr = hedging.hedging_premium(r, hp)
print(f"hedging-pressure information coefficient {pr['mean_ic']:+.3f} (t {pr['ic_t']:+.1f}) -- it predicts.")
print(f"null IC {hedging.hedging_premium(r0, hp0)['mean_ic']:+.3f} -- nothing.")

## 2 · So what? 💰

A genuine, positioning-driven risk premium would be gold: it's *structural* (someone always needs to hedge), uncorrelated with stocks, and readable from free public data. It's the backbone of commodity risk-premia funds. The desk's question is the usual one — is the toll still being paid on the markets you can actually trade, or did everyone crowd the booth until it closed?

## 3 · How we'd know 🔬

Three checks:

1. **Does hedging pressure predict returns?** The cross-sectional IC and the top-minus-bottom spread.
2. **Does the long-short factor pay**, net of cost?
3. **Is it robust / still alive** — across signal windows and sub-periods? And the **null:** no information ⇒ no premium.

**Mirage line:** the factor is flat-to-negative on the real tape, at every window.

## 4 · The teardown 🔧

### 4a · Premium tape vs the null
Run the hedging-pressure factor on both synthetic panels.

In [ ]:
for label, (rr, hh) in [('premium', (r, hp)), ('null', (r0, hp0))]:
    cmp = strategy.compare(rr, hh, cost_bps=10.0); pt = decompose.premium_tstat(rr, hh, cost_bps=10.0)
    print(f"{label:8s}: long-short Sharpe {cmp['long_short']['sharpe']:+.2f}, ann {cmp['long_short']['ann_return']:+.1%} "
          f"(HAC t {pt['t_stat']:+.1f}), turnover {cmp['turnover_ann']:.1f}x/yr")

### 4b · On the real market — the booth is empty
On real CFTC positioning + commodity futures (quoted from [`../docs/results.md`](../docs/results.md)):

- Hedging pressure predicts the next week with an IC of **-0.021** (*t* **-1.4**) — a *negative* top-minus-bottom spread (**-7.6%/yr**).
- The long-short factor **loses**: Sharpe **-0.41** (*t* **-1.4**), turnover **5.1×/yr**.
- And it's negative across *every* signal window, with sub-period Sharpes **-0.83 / -0.90 / +0.56** — gone for most of the modern sample.

## 5 · The verdict 🧾

- **Real mechanism** — control IC *t* > 11; decades of academic evidence.
- **Absent here** — real IC -0.021 (*t* -1.4), factor Sharpe -0.41.
- **Not a window artefact** — negative across the board.

> **Signal `WEAK` · Tradability `MIRAGE` · Premium today? `FADED`.** A toll that producers still pay *somewhere*, but not on the liquid futures you'd trade today.

## 6 · Could you trade it? 💸

- **The signal doesn't predict** on the modern tape — the long-short factor loses before you even sweat the costs.
- **Crowding is the likely culprit** — commodity risk-premia products and CTAs all read the same COT report; a structural premium that everyone harvests stops being one.
- **It may live in breadth** — the original studies used decades and dozens of contracts; our 12-commodity, 10-year window is thinner.

> Tradability **`MIRAGE`**; premium today? **`FADED`** — the worked complement ([`../docs/extension.md`](../docs/extension.md)) shows it's gone at every window.

## 7 · Going further 🚪

- **The robustness sweep** (beat 7 of the quants notebook + [`../docs/extension.md`](../docs/extension.md)): the factor is negative across signal windows and in both legs — not a tuning miss, an absence.
- **More breadth, more history.** The disaggregated COT (Producer/Merchant vs Managed Money) and a 30-contract, 1990s-on sample is where the academic premium lives; does it survive there where our liquid-12 window doesn't?
- **Combine with the futures basis / roll yield (§9.1).** Hedging pressure and term-structure carry are two reads on the same backwardation — does the *basis* still pay where positioning doesn't?

PRs welcome — widen the universe and history, or test the basis-based cousin.